# 🎓 MedScribe — 아이폰용 무료 클라우드 전사
**아이폰 메모리를 쓰지 않고** Google Colab 서버에서 Whisper를 실행합니다. 강의 녹음은 Google 서버로 업로드됩니다.

사용법: **런타임 → 런타임 유형 변경 → GPU(T4)** 를 선택할 수 있으면 선택 → **런타임 → 모두 실행** → 파일 선택 → 전사 완료 후 ZIP 저장.

- 한국어 중심 의학 강의에 맞춘 축어 전사이며 **요약하지 않습니다**.
- GPU를 못 받으면 CPU의 작은 모델로 실행합니다. 장시간 강의는 매우 느릴 수 있습니다.
- **Colab 무료 서비스는 GPU/이용시간이 보장되지 않으며 한도도 바뀔 수 있습니다.** 강의녹음 허용 여부와 개인정보 포함 여부를 확인하세요.


In [ ]:
!pip -q install "faster-whisper>=1.1.0"

In [ ]:
from google.colab import files
from pathlib import Path
upload = files.upload()
if not upload:
    raise RuntimeError("파일을 선택하지 않았어요. 업로드 셀을 다시 실행하세요.")
audio_path = Path(next(iter(upload.keys())))
print("녹음:", audio_path, "·", round(audio_path.stat().st_size / (1024**2), 1), "MB")

In [ ]:
from faster_whisper import WhisperModel
import torch, re, zipfile
from pathlib import Path

HAS_GPU = torch.cuda.is_available()
MODEL = "large-v3" if HAS_GPU else "small"
DEVICE = "cuda" if HAS_GPU else "cpu"
COMPUTE = "float16" if HAS_GPU else "int8"
LANGUAGE = "ko"  # 한국어 중심 강의, 자동 언어 감지는 None
TERMS = "GLP-1, SGLT2, DPP-4, insulin, glucagon, metformin, HbA1c, tirzepatide, semaglutide"
print(f"실행 방식: {MODEL} / {DEVICE}. 처음에는 모델 다운로드가 진행됩니다.")
model = WhisperModel(MODEL, device=DEVICE, compute_type=COMPUTE)
segments, info = model.transcribe(str(audio_path), language=LANGUAGE, task="transcribe", beam_size=5, vad_filter=False, initial_prompt=TERMS, condition_on_previous_text=False)

def t(sec):
    s=int(sec); return f"{s//3600:02}:{(s//60)%60:02}:{s%60:02}"
def srt_time(sec):
    ms=int(round(sec*1000)); h, ms=divmod(ms,3600000); m,ms=divmod(ms,60000); s,ms=divmod(ms,1000)
    return f"{h:02}:{m:02}:{s:02},{ms:03}"

stem=re.sub(r"[^\w가-힣-]+","_",audio_path.stem)[:80]
txt_path=Path(stem+"_전사.txt"); srt_path=Path(stem+"_자막.srt"); n=0
with txt_path.open("w",encoding="utf-8-sig") as txt, srt_path.open("w",encoding="utf-8-sig") as srt:
    for seg in segments:
        line=seg.text.strip()
        if not line: continue
        n+=1
        txt.write(f"[{t(seg.start)}] {line}\n")
        srt.write(f"{n}\n{srt_time(seg.start)} --> {srt_time(seg.end)}\n{line}\n\n")
        if n%10==0:
            txt.flush(); srt.flush(); print("처리:",t(seg.end),"/",t(info.duration))
print("전사 완료:",n,"개 구간")
zip_path=Path(stem+"_전사결과.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    z.write(txt_path,txt_path.name)
    z.write(srt_path,srt_path.name)
files.download(str(zip_path))